In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import joblib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

try:
    df = pd.read_csv('../clean/hanoi_aqi_cleaned.csv')
except:
    df = pd.read_csv('./hanoi_aqi_cleaned.csv')

df.columns = df.columns.str.strip().str.lower()
df['local_time'] = pd.to_datetime(df['local_time'])

FEATURES = [
    'co', 'no2', 'o3', 'pm10', 'pm25', 'so2',
    'clouds', 'precipitation', 'pressure',
    'relative_humidity', 'temperature', 'uv_index', 'wind_speed',
    'month', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'season'
]
TARGET = 'aqi'

# ══════════════════════════════════════════════════════
# TRAIN/TEST SPLIT THEO THỜI GIAN
# ══════════════════════════════════════════════════════
train = df[df['year'] < 2025].copy()
test  = df[df['year'] == 2025].copy()

X_train = train[FEATURES]
y_train = train[TARGET]

print(f"Train: {len(train):,} hàng (2022–2024)")
print(f"Test : {len(test):,} hàng (2025)")

# ══════════════════════════════════════════════════════
# TRAIN RANDOM FOREST
# ══════════════════════════════════════════════════════
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
print("\nĐang huấn luyện Random Forest...")
rf_model.fit(X_train, y_train)
print("✅ Hoàn thành!")

# ══════════════════════════════════════════════════════
# BẢNG KHUYẾN NGHỊ THEO NGƯỠNG AQI
# ══════════════════════════════════════════════════════
AQI_RULES = {
    'Good': {
        'range': (0, 50),
        'Hành động': '🍃 Không khí trong lành. Tăng cường thông gió, mở cửa sổ.',
        'Trẻ em':           '✅ Hoạt động bình thường',
        'Người già':        '✅ Hoạt động bình thường',
        'Bệnh hô hấp':      '✅ Hoạt động bình thường',
        'Người khỏe mạnh':  '✅ Hoạt động bình thường'
    },
    'Moderate': {
        'range': (51, 100),
        'Hành động': '🌤️ Có thể mở cửa sổ, nhưng nên hạn chế nếu nhà gần đường lớn.',
        'Trẻ em':           '⚠️ Hạn chế vận động mạnh',
        'Người già':        '⚠️ Tránh vận động mạnh',
        'Bệnh hô hấp':      '⚠️ Theo dõi triệu chứng',
        'Người khỏe mạnh':  '✅ Hoạt động bình thường'
    },
    'Unhealthy for sensitive groups': {
        'range': (101, 150),
        'Hành động': '🏠 Nên đóng bớt cửa sổ. Sử dụng máy lọc không khí nếu có thể.',
        'Trẻ em':           '🔶 Đeo khẩu trang',
        'Người già':        '🔶 Đeo khẩu trang N95',
        'Bệnh hô hấp':      '🔴 Tránh ra ngoài',
        'Người khỏe mạnh':  '⚠️ Hạn chế vận động mạnh'
    },
    'Unhealthy': {
        'range': (151, 200),
        'Hành động': '🚫 Đóng chặt cửa sổ để tránh bụi mịn xâm nhập. Bật máy lọc không khí.',
        'Trẻ em':           '🔴 Không nên ra ngoài',
        'Người già':        '🔴 Ở trong nhà',
        'Bệnh hô hấp':      '🔴 Ở trong nhà hoàn toàn',
        'Người khỏe mạnh':  '🔶 Đeo N95 nếu bắt buộc ra ngoài'
    },
    'Very Unhealthy': {
        'range': (201, 300),
        'Hành động': '🚨 Nguy hại! Đóng kín toàn bộ cửa sổ. Tránh mọi khe hở thông gió.',
        'Trẻ em':           '🚨 Ở trong nhà hoàn toàn',
        'Người già':        '🚨 Ở trong nhà hoàn toàn',
        'Bệnh hô hấp':      '🚨 Chuẩn bị thuốc cấp cứu',
        'Người khỏe mạnh':  '🔴 Tránh hoạt động ngoài trời'
    },
    'Hazardous': {
        'range': (301, 999),
        'Hành động': '🚨 Cực kỳ nguy hiểm! Đóng kín nhà cửa, sử dụng các biện pháp lọc khí tối đa.',
        'Trẻ em':           '🚨 Cấm ra ngoài',
        'Người già':        '🚨 Liên hệ y tế ngay',
        'Bệnh hô hấp':      '🚨 Cần hỗ trợ khẩn cấp',
        'Người khỏe mạnh':  '🚨 Không ra ngoài'
    }
}

def get_recommendation(aqi_value):
    for category, info in AQI_RULES.items():
        lo, hi = info['range']
        if lo <= aqi_value <= hi:
            return {
                'AQI':             aqi_value,
                'Mức độ':          category,
                'Hành động':       info['Hành động'],
                'Trẻ em':          info['Trẻ em'],
                'Người già':       info['Người già'],
                'Bệnh hô hấp':     info['Bệnh hô hấp'],
                'Người khỏe mạnh': info['Người khỏe mạnh'],
            }
    return None

# ══════════════════════════════════════════════════════
# DEMO — 6 CASE MINH HỌA (1 case mỗi mức)
# ══════════════════════════════════════════════════════
# Dùng aqi_category có sẵn trong file clean, không tính lại
cases = []
for cat in ['Good', 'Moderate', 'Unhealthy for sensitive groups',
            'Unhealthy', 'Very Unhealthy', 'Hazardous']:
    subset = test[test['aqi_category'] == cat]
    if len(subset) > 0:
        row = subset.iloc[0]
        pred_aqi = rf_model.predict(pd.DataFrame([row[FEATURES]]))[0]
        rec = get_recommendation(round(pred_aqi))
        if rec:
            cases.append({
                'Thời điểm':       str(row['local_time'])[:13] + 'h',
                'AQI thực tế':     row['aqi'],
                'AQI dự báo':      round(pred_aqi, 1),
                **{k: v for k, v in rec.items() if k != 'AQI'}
            })

print("\n" + "="*60)
print("HỆ THỐNG KHUYẾN NGHỊ — 6 CASE MINH HỌA")
print("="*60)
for i, case in enumerate(cases, 1):
    print(f"\nCase {i}: {case['Thời điểm']}  |  AQI thực tế={case['AQI thực tế']}  |  AQI dự báo={case['AQI dự báo']}")
    print(f"  Hành động      : {case['Hành động']}")
    print(f"  Mức độ         : {case['Mức độ']}")
    print(f"  Trẻ em         : {case['Trẻ em']}")
    print(f"  Người già      : {case['Người già']}")
    print(f"  Bệnh hô hấp    : {case['Bệnh hô hấp']}")
    print(f"  Người khỏe mạnh: {case['Người khỏe mạnh']}")

# ══════════════════════════════════════════════════════
#  LƯU BẢNG KHUYẾN NGHỊ TỔNG HỢP
# ══════════════════════════════════════════════════════
rec_table = []
for cat, info in AQI_RULES.items():
    lo, hi = info['range']
    rec_table.append({
        'Mức AQI':    f"{lo}–{'500+' if hi >= 999 else hi}",
        'Danh mục':   cat,
        'Trẻ em':     info['Trẻ em'],
        'Người già':  info['Người già'],
        'Bệnh hô hấp': info['Bệnh hô hấp'],
        'Khỏe mạnh':  info['Người khỏe mạnh'],
    })
pd.DataFrame(rec_table).to_csv('recommendation_table.csv', index=False, encoding='utf-8-sig')
print("\n Đã lưu: recommendation_table.csv")

# ══════════════════════════════════════════════════════
# LƯU MODEL
# ══════════════════════════════════════════════════════
joblib.dump(rf_model, 'best_model_rf_regression.pkl')
joblib.dump(FEATURES, 'regression_features.pkl')
print(" Đã lưu: best_model_rf_regression.pkl")
print(" Đã lưu: regression_features.pkl")

Train: 25,996 hàng (2022–2024)
Test : 4,345 hàng (2025)

Đang huấn luyện Random Forest...
✅ Hoàn thành!

HỆ THỐNG KHUYẾN NGHỊ — 6 CASE MINH HỌA

Case 1: 2025-01-26 10h  |  AQI thực tế=37.0  |  AQI dự báo=37.0
  Hành động      : 🍃 Không khí trong lành. Tăng cường thông gió, mở cửa sổ.
  Mức độ         : Good
  Trẻ em         : ✅ Hoạt động bình thường
  Người già      : ✅ Hoạt động bình thường
  Bệnh hô hấp    : ✅ Hoạt động bình thường
  Người khỏe mạnh: ✅ Hoạt động bình thường

Case 2: 2025-01-10 10h  |  AQI thực tế=100.0  |  AQI dự báo=100.4
  Hành động      : 🌤️ Có thể mở cửa sổ, nhưng nên hạn chế nếu nhà gần đường lớn.
  Mức độ         : Moderate
  Trẻ em         : ⚠️ Hạn chế vận động mạnh
  Người già      : ⚠️ Tránh vận động mạnh
  Bệnh hô hấp    : ⚠️ Theo dõi triệu chứng
  Người khỏe mạnh: ✅ Hoạt động bình thường

Case 3: 2025-01-03 16h  |  AQI thực tế=142.0  |  AQI dự báo=142.0
  Hành động      : 🏠 Nên đóng bớt cửa sổ. Sử dụng máy lọc không khí nếu có thể.
  Mức độ         : Unhea